In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

In [ ]:
from ppca import ppca_em, ppca_reconstruct_matrix, ppca_reconstruct_optimal
from random_nans import random_nans
from inject_random_missing_two_modes import inject_random_missing_two_modes
from select_retained_nPC import select_retained_nPC

def standardize(df_like, means, stds):
    return (df_like - means) / stds


def transform_SNR(X, b=0.0, rms_input=None, random_state=None):
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=float)
    
    if rms_input is None:
        rms_cols = np.sqrt(np.nanmean(X**2, axis=0))
    else:
        rms_cols = rms_input
        
    noise = b * rms_cols * rng.normal(0, 1, size=X.shape)
    X_noisy = np.where(np.isnan(X), np.nan, X + noise)
    
    return X_noisy, rms_cols


def missing_pattern(X, method="MCAR", fraction=0.2, random_state=42):
    if method == "MCAR":
        return random_nans(X.copy(), fraction=fraction, random_state=random_state)
    elif method == "mode_selected":
        return inject_random_missing_two_modes(X.copy(), fraction=fraction, random_state=random_state)
    else:
        raise ValueError("method must be 'MCAR' or 'mode_selected'")


def ppca_full_reconstruct(X_missing, W, sigma2, mu=None, return_latent=False, method='optimal'):
    if mu is None:
        mu = np.zeros(X_missing.shape[1])
    return ppca_reconstruct_matrix(
        X_missing, W, sigma2, mu, method=method, return_latent=return_latent
    )


def plot_combined_and_subplots(df, title_prefix, damage_start=320):
    x = np.arange(0, len(df))
    n_modes = df.shape[1]
    colors = plt.cm.tab10(np.linspace(0, 1, n_modes))

    mask_before = x <= damage_start
    mask_after = x >= damage_start  

    # --------------------------------------------------
    # Figure 1: 모든 모드를 한 플롯에 표시
    # --------------------------------------------------
    plt.figure(figsize=(14, 6))

    for i, col in enumerate(df.columns):
        y = df[col].values

        plt.plot(
            x[mask_before],
            y[mask_before],
            color=colors[i],
            linewidth=1.5,
            label=f"Mode {i+1}"
        )

        plt.plot(
            x[mask_after],
            y[mask_after],
            color='red',
            linewidth=1.5,
            alpha=0.8
        )

    plt.axvline(
        x=damage_start,
        color='black',
        linestyle='--',
        linewidth=1.3,
        label=f'Damage start ({damage_start})'
    )

    plt.title(f"After Auto-Scaling: All mode in plot", fontsize=14)
    plt.xlabel("Time sample")
    plt.ylabel("Natural frequency")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------
    # Figure 2: 모드별 subplot을 한 figure에 표시
    # --------------------------------------------------
    fig, axes = plt.subplots(n_modes, 1, figsize=(14, 2.8 * n_modes), sharex=True)

    if n_modes == 1:
        axes = [axes]

    for i, col in enumerate(df.columns):
        y = df[col].values
        ax = axes[i]

        # 손상 전
        ax.plot(
            x[mask_before],
            y[mask_before],
            color=colors[i],
            linewidth=1.5,
            label=f"Mode {i+1}"
        )

        # 손상 후
        ax.plot(
            x[mask_after],
            y[mask_after],
            color='red',
            linewidth=1.5,
            alpha=0.8,
            label='Damaged region' if i == 0 else None
        )

        # 손상 시작선
        ax.axvline(
            x=damage_start,
            color='black',
            linestyle='--',
            linewidth=1.2,
            label=f'Damage start ({damage_start})' if i == 0 else None
        )

        # 손상 전/후 평균선
        mean_before = np.mean(y[mask_before])
        mean_after = np.mean(y[mask_after])

        ax.axhline(
            mean_before,
            color=colors[i],
            linestyle=':',
            linewidth=1.2,
            alpha=0.8
        )
        ax.axhline(
            mean_after,
            color='red',
            linestyle=':',
            linewidth=1.2,
            alpha=0.8
        )

        # 평균값 텍스트
        drop = mean_after - mean_before
        drop_pct = (drop / mean_before) * 100

        ax.text(
            0.99, 0.88,
            f"Before mean = {mean_before:.6f}\nAfter mean = {mean_after:.6f}\nΔ = {drop:.6f} ({drop_pct:.3f}%)",
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
        )

        ax.set_ylabel(f"Mode {i+1}")
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best')

    axes[0].set_title(f"{title_prefix}: Mode-wise subplots(After Auto-Scaling)", fontsize=14)
    axes[-1].set_xlabel("Time sample")

    plt.tight_layout()
    plt.show()